# Hands-on 0: Setup and Installation

**Goal:** Install and verify the `dstparser` package.

This notebook will guide you through:
1. Installing dstparser in your home directory
2. Running tests to verify everything works
3. Understanding how dstparser works internally

## Step 1: Install dstparser

Install dstparser in your home directory under `~/ml/`.

### Option A: Copy from shared directory (Recommended for workshop)

```bash
mkdir -p ~/ml
cp -r /ceph/sharedfs/work/TAML2024/dstparser ~/ml/
```

### Option B: Clone from repository

```bash
mkdir -p ~/ml
cd ~/ml
git clone https://github.com/TA-DNN/dstparser.git
```

In [18]:
# Copy dstparser from shared directory
!mkdir -p ~/ml
!cp -r /ceph/sharedfs/work/TAML2024/dstparser ~/ml/

# Alternative: git clone (uncomment if needed)
# !cd ~/ml && git clone https://github.com/TA-DNN/dstparser.git

cp: cannot stat '/ceph/sharedfs/work/TAML2024/dstparser': No such file or directory


### Install the package

In [19]:
!pip install -e ~/ml/dstparser

Obtaining file:///home/antonpr/ml/dstparser
  Installing build dependencies ... -done
  Checking if build backend supports build_editable ... one
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Getting requirements to build editable ... -done
  Preparing editable metadata (pyproject.toml) ... one
  Preparing editable metadata (pyproject.toml) ... -done
  Building editable for dstparser (pyproject.toml) ... one
  Building editable for dstparser (pyproject.toml) ... -done
  Created wheel for dstparser: filename=dstparser-0.1.0-0.editable-py3-none-any.whl size=1907 sha256=52327480abccd12d9d1883d72a438e8f0c06bcea0f71bd70596232434ddc97dd
  Stored in directory: /tmp/pip-ephem-wheel-cache-2ckci1bh/wheels/39/00/a7/f3fd34665d4b199de74c1c00da68c71ccf4da0d7395935deeb
Successfully built dstparser
done
  Created wheel for dstparser: filename=dstparser-0.1.0-0.editable-py3-none-any.whl size=1907 sha256=52327480abccd12d9d1883d72a438e8f

## Step 2: Verify Installation

**Task:** Check that dstparser was installed correctly.

Run the cell below to import dstparser and check its version:

In [27]:
import dstparser

print(f"✓ dstparser successfully imported!")
print(f"  Version: {dstparser.__version__}")
print(f"  Location: {dstparser.__file__}")

✓ dstparser successfully imported!
  Version: 0.1.0
  Location: /home/antonpr/ml/dstparser/src/dstparser/__init__.py


## Step 3: Update Configuration

Configure dstparser to use the shared directory with pre-compiled binaries.

Edit `~/ml/dstparser/src/dstparser/paths.py` and change these two lines:

```python
root_dir = "/ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019"
openssl10_fix_dir = "/ceph/sharedfs/work/TAML2024/benMC/install"
```

The `root_dir` tells dstparser where to find:
- The `sditerator` binary (in the `bin/` subdirectory)
- The environment setup script (`sdanalysis_env.sh`)

The `openssl10_fix_dir` patches problems with openssl10 libraries from `root`. 

## Step 4: Understanding Data Layout

### Data Directory Structure

DST files are organized as:
```
/ceph/sharedfs/work/TAML2024/benMC/tasdmc_dstbank/
└── qgsii04proton/080417_160603/Em1_bsdinfo/
    ├── DAT000000_gea.rufldf.dst.gz
    ├── DAT000001_gea.rufldf.dst.gz
    ├── ...
    └── DAT*_xmax.txt  (text files with Xmax values)
```

**DST files (.dst.gz):**
- Binary format containing detector data
- Event triggers, waveforms, timing
- Reconstructed parameters (energy, direction, core)

**Xmax files (*_xmax.txt):**
- Text files with shower maximum depth
- Read together with DST files to get Xmax information
- Not contained in DST files themselves

## Step 5: Update Test File Data Path

Before running the test, update the data path in `~/ml/dstparser/tests/test_parser.py`.

Find and change the `xmax_dir` path and dst file to read (`dst_file`):
```python
xmax_dir = "/ceph/sharedfs/work/TAML2024/benMC/tasdmc_dstbank/qgsii04proton/080417_160603/Em1_bsdinfo"

dst_file = "/ceph/sharedfs/work/TAML2024/benMC/tasdmc_dstbank/qgsii04proton/080417_160603/Em1_bsdinfo/DAT000000_gea.rufldf.dst.gz"
```

In [21]:
# Run test with data from shared directory
!python tests/test_parser.py

python: can't open file '/ceph/work/SATORI/antonpr/ml/benMC/sdanalysis_2019/handson/tests/test_parser.py': [Errno 2] No such file or directory


## Step 6: What is dstparser?

Now that you have dstparser working, let's understand how it works internally.

### dstparser is a Python wrapper

The `dstparser` package does NOT parse DST files directly in Python. Instead:

1. **It calls compiled C++ executables** - Specifically `sditerator` tools
2. **These executables read DST files** - The C++ code handles the binary format  
3. **Outputs are captured** - Python subprocess captures text output
4. **Data is parsed** - Python converts text to numpy arrays and dictionaries

### Dependency Chain:

```
Your Python code
    ↓
dstparser (Python wrapper)
    ↓  
subprocess calls  
    ↓
sditerator_*.run (Compiled C++ binary)
    ↓
dst2k-ta libraries
    ↓
ROOT framework
```

### How it works:

```
Python: parse_dst_file("file.dst.gz")
           ↓
dstparser calls subprocess
           ↓
C++ binary: sditerator_add_standard_recon_v2.run
           ↓
Reads DST file → Outputs text
           ↓
dstparser parses text → Returns Python dictionary
```

The actual DST file parsing is done by **sditerator**, a C++ program that:
- Reads binary DST format
- Extracts event data
- Applies reconstruction
- Outputs results as text

## Step 8: Using sditerator Directly

You can run sditerator directly to see its output.

**Important:** To use any sdanalysis tool, you must first source the environment:
```bash
source /ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sdanalysis_env.sh
```

Let's run sditerator on a data file:

```bash
/ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/bin/sditerator_add_standard_recon_v2.run /ceph/sharedfs/work/TAML2024/benMC/tasdmc_dstbank/qgsii04proton/080417_160603/Em1_bsdinfo/DAT000000_gea.rufldf.dst.gz | head -50
```

**What you see:**
- Text output with numerical data
- Each line represents different information (event ID, energy, positions, etc.)
- This is what dstparser parses and converts to Python dictionaries

### Other Tools

**dstdump** - Shows detailed DST file structure. Also requires sourcing environment first:
```bash
source /ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sdanalysis_env.sh
dstdump.run /ceph/sharedfs/work/TAML2024/benMC/tasdmc_dstbank/qgsii04proton/080417_160603/Em1_bsdinfo/DAT000000_gea.rufldf.dst.gz
```

## Step 9: Shared Installation Contents

The pre-compiled installation contains:

```
/ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/
├── bin/                  # Compiled binaries
│   ├── sditerator_add_standard_recon_v2.run
│   └── dstdump.run
├── sditerator/src/       # Source code
│   └── sditerator_cppanalysis.cpp
└── sdanalysis_env.sh     # Environment setup
```

## Step 10: Examining Source Code

The main sditerator code is at:
```
/ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sditerator/src/sditerator_add_standard_recon_v2.run
```

In [26]:
# View the beginning of the source code to see includes and structure
!head -100 /ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sditerator/src/sditerator_add_standard_recon_v2.run

head: cannot open '/ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sditerator/src/sditerator_add_standard_recon_v2.run' for reading: No such file or directory


### Building sditerator

**The source codes of sditerator:**
```
sdanalysis_2019/sditerator/src/
```

**How to edit/compile sditerator (in your environment):**

1. Source the environment:
```bash
source sdanalysis_2019/sdanalysis_env.sh
```

2. Edit the source file:
```bash
# Edit sdanalysis_2019/sditerator/src/sditerator_cppanalysis_add_standard_recon_v2.cpp
# The output format is written in the cpp file
```

3. Compile:
```bash
cd sdanalysis_2019/sditerator
./rebuild.sh
```

**Output:** `sdanalysis_2019/bin/sditerator_add_standard_recon_v2.run` will be built

## Troubleshooting

**If test fails:**
1. Check `paths.py` configuration
2. Verify binary exists: `ls -l /ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/bin/sditerator_add_standard_recon_v2.run`
3. Check data file path in `test_parser.py`

**If binary doesn't run:**
- Source environment first: `source /ceph/sharedfs/work/TAML2024/benMC/install/sdanalysis_2019/sdanalysis_env.sh`